# 🛡️ CyberTech NeuralShield — Google Colab Çok Modelli Benchmark

Bu Colab çalışma alanı; **TÜBİTAK / EVREN H200** altyapısında API üzerinden test edilen modellere ek olarak,
Hugging Face ekosistemindeki açık ağırlıklı amiral gemisi modelleri (**Llama-3.3-70B**, **Qwen-2.5-72B**, **Trendyol-LLM**, **Mistral-7B**)
aynı kademeli saldırı havuzuyla (Tier 1 - Tier 5) GPU üzerinde test etmek için tasarlanmıştır.

### 🎯 Değerlendirme Modları:
1. **Kalkan KAPALI (Shield OFF):** Çıplak modelin kanarya sızıntı oranı (Attack Success Rate - ASR %)
2. **Kalkan AÇIK (Shield ON):** CyberTech NeuralShield-v5 (ONNX INT8 ~12ms) devredeyken saldırının kapıda engellenmesi


In [ ]:
# @title 1. GPU ve Donanım Kontrolü
!nvidia-smi


In [ ]:
# @title 2. Gerekli Kütüphanelerin Yüklenmesi
!pip install -q transformers accelerate bitsandbytes onnxruntime torch python-dotenv


In [ ]:
# @title 3. Proje Deposunu Klonlama veya Güncelleme
import os
if not os.path.exists('ai-soc-agent-lab'):
    !git clone https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git
%cd ai-soc-agent-lab
!git pull
import sys
sys.path.insert(0, os.path.abspath('backend'))
print('✅ Çalışma dizini:', os.getcwd())


In [ ]:
# @title 4. Hedef Modeli Seçin ve Testi Başlatın (Colab A100 80GB / T4 Uyumlu)
# @markdown **Siklet 1 (Küçük 2B-3B)** | **Siklet 2 (Orta 7B-32B & Türkçe)** | **Sansürsüz (Abliterated)** | **Siklet 3 (Büyük 70B):**
TARGET_MODEL = "Trendyol/Trendyol-LLM-7b-chat-v1.0" #@param ["Qwen/Qwen2.5-3B-Instruct", "meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-2b-it", "microsoft/Phi-3.5-mini-instruct", "Trendyol/Trendyol-LLM-7b-chat-v1.0", "mistralai/Mistral-Nemo-Instruct-2407", "Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-32B-Instruct", "google/gemma-2-27b-it", "huihui-ai/Qwen2.5-14B-Instruct-abliterated", "failspy/Meta-Llama-3-8B-Instruct-abliterated-v3", "meta-llama/Llama-3.3-70B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"]
DATASET_PATH = "data/colab_e2e_suite_stratified.json" #@param ["data/colab_e2e_suite_stratified.json", "data/appsec_benchmark_2000.json", "data/evren_unified_benchmark_25.json", "data/evren_benchmark_30_independent.json"]
HF_TOKEN = "" #@param {type:"string"}
USE_4BIT = True #@param {type:"boolean"}

cmd = f"python scripts/colab/run_colab_benchmark.py --model {TARGET_MODEL} --dataset {DATASET_PATH}"
if USE_4BIT:
    cmd += " --load-in-4bit"
if HF_TOKEN:
    cmd += f" --hf-token {HF_TOKEN}"

print(f"🚀 Benchmark Başlatılıyor: {cmd}")
!{cmd}


In [ ]:
# @title 5. Çıktı Grafiğini Çizdirin (Kalkan KAPALI vs AÇIK ASR %)
import json, glob, matplotlib.pyplot as plt

results_files = glob.glob('data/colab_benchmark_*.json')
if results_files:
    latest = max(results_files, key=os.path.getctime)
    with open(latest, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    model_name = data.get('model', 'Hedef Model')
    raw_asr = data.get('raw_asr_pct', 0)
    shield_asr = data.get('shield_asr_pct', 0)
    block_rate = data.get('block_rate_pct', 0)
    
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    bars = ax.bar(['Kalkan KAPALI (Raw LLM)', 'Kalkan AÇIK (NeuralShield)'], [raw_asr, shield_asr], color=['#e74c3c', '#2ecc71'], width=0.5)
    ax.set_ylabel('Saldırı Başarı Oranı (ASR %)', fontsize=12)
    ax.set_title(f'{model_name} - Hacklenme Riski Karşılaştırması', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f'%{yval:.1f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('colab_benchmark_comparison.png')
    plt.show()
    print('✅ Grafik kaydedildi: colab_benchmark_comparison.png')
else:
    print('⚠️ Henüz tamamlanmış benchmark verisi bulunamadı.')
